# **Neural Machine Translation with Language Models**
In this lab, we build a complete Neural Machine Translation (NMT) pipeline using a multilingual sequence-to-sequence (Seq2Seq) language model. The goal is to fine-tune a pre-trained BART-based model on a bilingual Tay–Vietnamese dataset and evaluate its translation quality using BLEU and METEOR metrics.

# **Install libraries & import modules**
This cell installs all required packages (Transformers, Datasets, Torch, Evaluate).
These provide tools for tokenization, model loading, training, BLEU scoring, etc.

In [ ]:
!pip install sacrebleu transformers datasets torch evaluate accelerate openpyxl sentencepiece matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00


# **Define model checkpoint, dataset name, and sequence lengths**

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running on Colab:", IN_COLAB)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd
import numpy as np
import re
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)
import evaluate
import os

MODEL_CKPT = "FiveC/BartTay"

DATA_DIR = "/content/drive/MyDrive/Lab6-BARTpho" if IN_COLAB else "."
TRAIN_PATH = os.path.join(DATA_DIR, "train.xlsx")
TEST_PATH  = os.path.join(DATA_DIR, "test.xlsx")

train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

# **Data Cleaning Functions**
This cell defines clean_text() to remove:

+ special characters

+ newlines, tabs

+ extra whitespace

Clean data helps the model learn better and prevents noise from damaging the translation quality.

In [ ]:
train_df = train_df.dropna(subset=["tay", "viet"])
test_df  = test_df.dropna(subset=["tay", "viet"])

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return None
    text = re.sub(r'\\n|\|\\|[\n\r\t]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Áp dụng cleaning cho từng dòng
train_df["tay"]  = train_df["tay"].astype(str).apply(clean_text)
train_df["viet"] = train_df["viet"].astype(str).apply(clean_text)

test_df["tay"]  = test_df["tay"].astype(str).apply(clean_text)
test_df["viet"] = test_df["viet"].astype(str).apply(clean_text)

print(train_df.head())
print(test_df.head())

# **Load tokenizer and model**

In [ ]:
print(f"Loading tokenizer & model from {MODEL_CKPT} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CKPT)

# **Preprocessing function**
This function tokenizes Tay sentences as inputs and Vietnamese sentences as targets.

Key steps:

+ truncate & pad inputs

+ tokenize targets using `text_target=` and replace padding in labels with -100

+ attach labels to model inputs

The output format matches what Seq2SeqTrainer expects.

In [ ]:
def preprocess_function(examples):
    inputs = examples["tay"]
    targets = examples["viet"]

    # Tokenize input + target (text_target replaces the removed as_target_tokenizer())
    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=MAX_SOURCE_LENGTH,
        max_target_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    # Ignore padding tokens in the loss
    model_inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in seq]
        for seq in model_inputs["labels"]
    ]
    return model_inputs


In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

In [ ]:
cols_to_remove = ["__index_level_0__"]
train_dataset = train_dataset.remove_columns([c for c in cols_to_remove if c in train_dataset.column_names])
test_dataset  = test_dataset.remove_columns([c for c in cols_to_remove if c in test_dataset.column_names])

In [ ]:
train_dataset = train_dataset.map(preprocess_function, batched=True)
test_dataset  = test_dataset.map(preprocess_function, batched=True)

# **Define SacreBLEU metric**

In [ ]:
import sacrebleu as sb

def postprocess_text(preds, labels):
    """
    - Loại bỏ token đặc biệt, strip(), ...
    - pred: list[str], label: list[str].
    """
    preds = [pred.strip() for pred in preds]
    labels = [lbl.strip() for lbl in labels]
    return preds, labels

def compute_metrics(eval_preds):
    """
    eval_preds = (predictions, labels)
    """
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_str, label_str = postprocess_text(pred_str, label_str)

    bleu = sb.corpus_bleu(pred_str, [label_str]).score
    chrf = sb.corpus_chrf(pred_str, [label_str]).score
    return {"sacrebleu": bleu, "chrf": chrf}

# **Configure training arguments**

In [ ]:
import torch

BATCH_SIZE = 16

training_args = Seq2SeqTrainingArguments(
    f"BartTayFinal",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=100,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="sacrebleu",
    greater_is_better=True,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

In [ ]:

from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
from accelerate import Accelerator
accelerator = Accelerator()

print("Starting Training ...")
trainer.train()

**Train the model using three different optimizers and compare their impact on translation performance:**

+ AdamW (default)

+ Adafactor (recommended for Seq2Seq models)

+ RMSprop

**For each optimizer:**

+ Keep all hyperparameters the same except for the optimizer

+ Train for the same number of epochs

+ Evaluate the model on the test set

+ Record and compare BLEU, chrF, and training stability

**Analysis should consider:**

+ Which optimizer achieves the best performance

+ Which optimizer converges faster

+ Which optimizer is more stable, especially for low-resource languages like Tay

# **Optimizer comparison: AdamW vs Adafactor vs RMSprop**
Each run starts from the same pre-trained checkpoint `FiveC/BartTay`, uses identical hyperparameters (learning rate, batch size, epochs, weight decay, seed) and only changes the optimizer. After training, the model is evaluated on the test set with BLEU and chrF, and the per-epoch eval loss/BLEU history is kept to judge convergence speed and stability.

In [ ]:
import time, gc
import torch
import pandas as pd
from torch.optim import RMSprop
from transformers.optimization import Adafactor, get_linear_schedule_with_warmup
from transformers import DataCollatorForSeq2Seq, set_seed

NUM_EPOCHS = 5
LR = 2e-5
SEED = 42

def build_args(run_name, optim="adamw_torch"):
    return Seq2SeqTrainingArguments(
        output_dir=f"BartTay_{run_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        logging_steps=50,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        load_best_model_at_end=True,
        metric_for_best_model="sacrebleu",
        greater_is_better=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        optim=optim,
        seed=SEED,
        report_to="none",
    )

def make_optimizer(name, model, num_training_steps):
    """Return (optimizer, scheduler) or None to let Trainer build the default AdamW."""
    if name == "AdamW":
        return None
    decay = [p for n, p in model.named_parameters() if p.requires_grad and not any(k in n for k in ["bias", "LayerNorm", "layer_norm"])]
    no_decay = [p for n, p in model.named_parameters() if p.requires_grad and any(k in n for k in ["bias", "LayerNorm", "layer_norm"])]
    groups = [{"params": decay, "weight_decay": 0.01}, {"params": no_decay, "weight_decay": 0.0}]
    if name == "Adafactor":
        opt = Adafactor(groups, lr=LR, scale_parameter=False, relative_step=False, warmup_init=False)
    elif name == "RMSprop":
        opt = RMSprop(groups, lr=LR, alpha=0.99, eps=1e-8)
    else:
        raise ValueError(name)
    sched = get_linear_schedule_with_warmup(opt, 0, num_training_steps)
    return opt, sched

def run_experiment(opt_name):
    set_seed(SEED)
    m = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CKPT)
    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=m)
    args = build_args(opt_name)
    steps_per_epoch = int(np.ceil(len(train_dataset) / (BATCH_SIZE * max(1, args.gradient_accumulation_steps))))
    custom = make_optimizer(opt_name, m, steps_per_epoch * NUM_EPOCHS)
    tr = Seq2SeqTrainer(
        model=m,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=collator,
        compute_metrics=compute_metrics,
        optimizers=custom if custom is not None else (None, None),
    )
    t0 = time.time()
    tr.train()
    train_time = time.time() - t0
    final = tr.evaluate()
    hist = pd.DataFrame(tr.state.log_history)
    train_loss = hist.dropna(subset=["loss"])[["step", "loss"]] if "loss" in hist else pd.DataFrame()
    eval_hist = hist.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_sacrebleu", "eval_chrf"]]
    del tr, m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "optimizer": opt_name,
        "BLEU": final["eval_sacrebleu"],
        "chrF": final["eval_chrf"],
        "eval_loss": final["eval_loss"],
        "train_time_s": train_time,
        "train_loss_std": float(train_loss["loss"].diff().abs().mean()) if len(train_loss) > 1 else float("nan"),
    }, eval_hist, train_loss

results, eval_histories, train_losses = [], {}, {}
for opt_name in ["AdamW", "Adafactor", "RMSprop"]:
    print(f"\n========== Training with {opt_name} ==========")
    res, eh, tl = run_experiment(opt_name)
    results.append(res)
    eval_histories[opt_name] = eh
    train_losses[opt_name] = tl
    print(res)

results_df = pd.DataFrame(results).set_index("optimizer")
results_df

**Per-epoch evaluation history (convergence)**

In [ ]:
for name, eh in eval_histories.items():
    print(f"--- {name} ---")
    print(eh.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for name, tl in train_losses.items():
    if len(tl):
        axes[0].plot(tl["step"], tl["loss"], label=name)
for name, eh in eval_histories.items():
    axes[1].plot(eh["epoch"], eh["eval_loss"], marker="o", label=name)
    axes[2].plot(eh["epoch"], eh["eval_sacrebleu"], marker="o", label=name)
axes[0].set_title("Training loss"); axes[0].set_xlabel("step")
axes[1].set_title("Eval loss"); axes[1].set_xlabel("epoch")
axes[2].set_title("Eval BLEU"); axes[2].set_xlabel("epoch")
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# **Analysis**
Use the table and plots above to answer:

+ **Best performance**: the optimizer with the highest test BLEU / chrF in `results_df`.
+ **Fastest convergence**: the optimizer whose eval BLEU reaches its plateau (or eval loss drops) in the fewest epochs in the per-epoch history.
+ **Stability**: the optimizer with the smoothest training-loss curve (lowest `train_loss_std`, i.e. mean absolute change between consecutive logged losses) and no BLEU drops between epochs. For a low-resource language like Tay (~4.9k training pairs), a stable optimizer avoids overfitting spikes and catastrophic forgetting of the pre-trained knowledge.